In [ ]:
import os
import shutil
import torch
from ogb.linkproppred import PygLinkPropPredDataset

# Patch torch.load BEFORE importing OGB to handle weights_only default
_torch_load_orig = torch.load.__wrapped__ if hasattr(torch.load, '__wrapped__') else torch.load

def torch_load_patched(f, *args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _torch_load_orig(f, *args, **kwargs)

torch.load = torch_load_patched

# Clean cached dataset to force fresh load
dataset_root = 'dataset'
if os.path.exists(dataset_root):
    shutil.rmtree(dataset_root)
    print(f"Cleaned {dataset_root}")

# Load the dataset
dataset_name = 'ogbl-ddi'
dataset = PygLinkPropPredDataset(name = "ogbl-ddi", root = '/data/giobbi')
print(f'The {dataset_name} dataset has {len(dataset)} graph(s).')

Cleaned dataset


Downloaded 0.04 GB: 100%|██████████| 46/46 [00:09<00:00,  4.76it/s]


Extracting dataset/ddi.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 48.51it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 3718.35it/s]

Saving...
The ogbl-ddi dataset has 1 graph(s).



Done!


In [19]:
# This graph only contains training edges
ddi_graph = dataset[0]

print(f'DDI graph object: {ddi_graph}')
print(f'Number of nodes |V|: {ddi_graph.num_nodes}')
print(f'Number of (training) edges |E|: {ddi_graph.num_edges}')
print(f'Is undirected: {ddi_graph.is_undirected()}')
# Note that since the graph is undirected, PyG includes both (u, v) and (v, u) as edges
print(f'Average node degree: {ddi_graph.num_edges / ddi_graph.num_nodes:.2f}')
print(f'Number of node features: {ddi_graph.num_node_features}')
print(f'Has isolated nodes: {ddi_graph.has_isolated_nodes()}')
print(f'Has self-loops: {ddi_graph.has_self_loops()}')

DDI graph object: Data(num_nodes=4267, edge_index=[2, 2135822])
Number of nodes |V|: 4267
Number of (training) edges |E|: 2135822
Is undirected: True
Average node degree: 500.54
Number of node features: 0
Has isolated nodes: False
Has self-loops: False


In [23]:
type(ddi_graph)

torch_geometric.data.data.Data

In [20]:
import pandas as pd

ddi_df = pd.read_csv('/data/giobbi/ogbl_ddi/mapping/ddi_description.csv.gz', compression='gzip')
node_df = pd.read_csv('/data/giobbi/ogbl_ddi/mapping/nodeidx2drugid.csv.gz', compression='gzip')

print(ddi_df.head())
print(node_df.head())

  first drug id first drug name second drug id      second drug name  \
0       DB00001       Lepirudin        DB06605              Apixaban   
1       DB00001       Lepirudin        DB06695  Dabigatran etexilate   
2       DB00001       Lepirudin        DB01254             Dasatinib   
3       DB00001       Lepirudin        DB01609           Deferasirox   
4       DB00001       Lepirudin        DB01586  Ursodeoxycholic acid   

                                         description  
0  Apixaban may increase the anticoagulant activi...  
1  Dabigatran etexilate may increase the anticoag...  
2  The risk or severity of bleeding and hemorrhag...  
3  The risk or severity of gastrointestinal bleed...  
4  The risk or severity of bleeding and bruising ...  
   node idx  drug id
0         0  DB00001
1         1  DB00002
2         2  DB00004
3         3  DB00005
4         4  DB00006


In [24]:
node_df

,node idx,drug id
0,0,DB00001
1,1,DB00002
2,2,DB00004
3,3,DB00005
4,4,DB00006
...,...,...
4262,4262,DB15595
4263,4263,DB15598
4264,4264,DB15617
4265,4265,DB15623


In [21]:
ddi_df

,first drug id,first drug name,second drug id,second drug name,description
0,DB00001,Lepirudin,DB06605,Apixaban,Apixaban may increase the anticoagulant activi...
1,DB00001,Lepirudin,DB06695,Dabigatran etexilate,Dabigatran etexilate may increase the anticoag...
2,DB00001,Lepirudin,DB01254,Dasatinib,The risk or severity of bleeding and hemorrhag...
3,DB00001,Lepirudin,DB01609,Deferasirox,The risk or severity of gastrointestinal bleed...
4,DB00001,Lepirudin,DB01586,Ursodeoxycholic acid,The risk or severity of bleeding and bruising ...
...,...,...,...,...,...
2669759,DB15657,Ala-geninthiocin,DB13347,Diphenadione,The risk or severity of bleeding can be increa...
2669760,DB15657,Ala-geninthiocin,DB13451,Tioclomarol,The risk or severity of bleeding can be increa...
2669761,DB15657,Ala-geninthiocin,DB14055,(S)-Warfarin,The risk or severity of bleeding can be increa...
2669762,DB15657,Ala-geninthiocin,DB00581,Lactulose,The therapeutic efficacy of Lactulose can be d...


In [9]:
split_edges = dataset.get_edge_split()
train_edges, valid_edges, test_edges = split_edges['train'], split_edges['valid'], split_edges['test']